# Project Atlas — COLMAP + gsplat (Colab T4)

**What this notebook does:**
1. Upload `frames_for_colab.zip` (extracted by `capture.py` locally)
2. Run COLMAP SfM (camera poses + sparse point cloud)
3. Train Gaussian Splat via nerfstudio
4. Export `.splat` file → download as `splat_result.zip`

**After download:** unzip into `data/scans/{scan_id}/splat/` locally

**Expected time:** ~15 min total on free T4 for a single room (30–60 frames)

---
**Before running:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Cell 1 — Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime → Change runtime type → T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 2 — Install dependencies
print('Installing COLMAP...')
!apt-get install -y colmap -q

print('Installing nerfstudio + gsplat...')
!pip install nerfstudio -q

# Verify
!colmap --version
import nerfstudio
print('nerfstudio:', nerfstudio.__version__)

In [ ]:
# Cell 3 — Upload frames zip
# Upload: data/scans/{scan_id}/frames_for_colab.zip  (created by capture.py locally)
from google.colab import files
import zipfile, os, shutil

print('Upload frames_for_colab.zip ...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

images_dir = '/content/images'
os.makedirs(images_dir, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/frames_raw')

# Flatten: move all .jpg to /content/images/
for root, _, files_ in os.walk('/content/frames_raw'):
    for f in files_:
        if f.endswith('.jpg'):
            shutil.copy(os.path.join(root, f), os.path.join(images_dir, f))

frame_count = len(os.listdir(images_dir))
print(f'Extracted {frame_count} frames to {images_dir}')

In [ ]:
# Cell 4 — Run COLMAP (SfM)
import subprocess, os

db_path     = '/content/colmap.db'
sparse_dir  = '/content/sparse'
ns_data_dir = '/content/ns_data'
os.makedirs(sparse_dir, exist_ok=True)
os.makedirs(ns_data_dir, exist_ok=True)

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('STDERR:', r.stderr[-1500:])
        raise RuntimeError(f'Failed: {cmd[0]} {cmd[1]}')
    return r

print('COLMAP: feature extraction...')
run(['colmap', 'feature_extractor',
     '--database_path', db_path,
     '--image_path', '/content/images',
     '--ImageReader.single_camera', '1'])

print('COLMAP: exhaustive matching...')
run(['colmap', 'exhaustive_matcher', '--database_path', db_path])

print('COLMAP: sparse reconstruction...')
run(['colmap', 'mapper',
     '--database_path', db_path,
     '--image_path', '/content/images',
     '--output_path', sparse_dir])

print('COLMAP done. Sparse models found:')
!ls /content/sparse/

In [ ]:
# Cell 5 — Convert to nerfstudio format
import subprocess

print('Converting COLMAP → nerfstudio format...')
r = subprocess.run([
    'ns-process-data', 'colmap',
    '--data', '/content/images',
    '--colmap-model-path', '/content/sparse/0',
    '--output-dir', '/content/ns_data',
    '--skip-colmap',  # already ran COLMAP above
], capture_output=True, text=True)

if r.returncode != 0:
    # Fallback: run ns-process-data with its own colmap
    print('Fallback: running ns-process-data with built-in COLMAP...')
    subprocess.run([
        'ns-process-data', 'video',
        '--data', '/content/images',
        '--output-dir', '/content/ns_data',
    ], check=True)

print('ns_data ready:')
!ls /content/ns_data/

In [ ]:
# Cell 6 — Train Gaussian Splat
import subprocess

output_dir = '/content/splat_output'

print('Training splatfacto (7000 iterations, ~10 min on T4)...')
r = subprocess.run([
    'ns-train', 'splatfacto',
    '--data', '/content/ns_data',
    '--output-dir', output_dir,
    '--max-num-iterations', '7000',
    '--pipeline.model.sh-degree', '0',
], capture_output=True, text=True)

if r.returncode != 0:
    print('STDOUT:', r.stdout[-2000:])
    print('STDERR:', r.stderr[-2000:])
    raise RuntimeError('Training failed')

print('Training complete.')
!find /content/splat_output -name '*.ckpt' | head -5

In [ ]:
# Cell 7 — Export .splat file
import subprocess, glob

configs = glob.glob('/content/splat_output/**/config.yml', recursive=True)
if not configs:
    raise FileNotFoundError('No config.yml found. Did training complete?')
config_path = configs[0]
print(f'Using config: {config_path}')

export_dir = '/content/splat_export'
subprocess.run([
    'ns-export', 'gaussian-splat',
    '--load-config', config_path,
    '--output-dir', export_dir,
], check=True)

print('Export done:')
!ls -lh /content/splat_export/

In [ ]:
# Cell 8 — Download result
import zipfile, os
from google.colab import files

zip_out = '/content/splat_result.zip'
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, filenames in os.walk('/content/splat_export'):
        for fname in filenames:
            fpath = os.path.join(root, fname)
            zf.write(fpath, os.path.relpath(fpath, '/content/splat_export'))

print('Downloading splat_result.zip...')
print('After download: unzip into data/scans/{scan_id}/splat/ locally')
files.download(zip_out)